```python
from json import loads
import numpy as np
def m():
    global m
    m = {"。": 0}
    for i in "train", "valid":
        for i in open(f"../tmp/news2016zh_{i}.json"):
            for i, j in loads(i).items():
                if j and i not in ("news_id", "time"):
                    for i in j:
                        yield m.setdefault(i, len(m))
                    if 19968 <= ord(j[-1]) <= 40869:
                        yield 0
a = np.fromiter(m(), "H")
```

```python
m = Learner(
    DataLoaders(Dat(64, 16*5, 50000), Dat(64, 16*5, 500000)),
    Sequential(
        Embedding(21482, 32),
        PyramidAutoEncoder(
            Sequential(View(64), Linear(64, 32, 0), BatchNorm1d(32, track_running_stats=0), Swish()),
            Sequential(Linear(32, 32, 0), BatchNorm1d(32, track_running_stats=0), Swish(), Linear(32, 32)),
            Sequential(BatchNorm1d(32, track_running_stats=0), Swish(), Linear(32, 64, 0), View(32)),
            Sequential(Linear(32, 32, 0), BatchNorm1d(32, track_running_stats=0), Swish(), Linear(32, 21482)),
        ),
    ).cuda(),
    MultiCrossEntropyLoss(),
    metrics=map(Accuracy, range(6)),
    cbs=ShowGraphCallback,
)
```

```python
m = Learner(
    DataLoaders(Dat(64, 16*5, 50000), Dat(64, 16*5, 500000)),
    Sequential(
        Embedding(21482, 32),
        PyramidAutoEncoder(
            Sequential(View(64), Linear(64, 32, 0), BatchNorm1d(32, track_running_stats=0), Swish()),
            Sequential(Linear(32, 32, 0), BatchNorm1d(32, track_running_stats=0), Swish(), Linear(32, 32)),
            Sequential(BatchNorm1d(32, track_running_stats=0), Swish(), Linear(32, 64, 0), View(32)),
            Sequential(Linear(32, 32, 0), BatchNorm1d(32, track_running_stats=0), Swish(), Linear(32, 21482)),
        ),
    ).cuda(),
    MultiCrossEntropyLoss(),
    metrics=map(Accuracy, range(6)),
    cbs=ShowGraphCallback,
)
```

In [ ]:
from fastai.vision.all import *
from torch.nn import *

torch.cuda.set_device(1)


class Dat:
    a = torch.from_numpy(np.memmap(f"../tmp/news2016zh.H", "h"))

    def __init__(s, w, b, n=1):
        s.a = s.a.as_strided([len(s.a) - w + 1, w], [1, 1])
        s.b = b
        s.n = n

    def __len__(s):
        return len(s.a) // (s.a.size(1) * s.b * s.n)

    def __iter__(s):
        for x in range(len(s)):
            x = s.a[torch.randint(len(s.a), [s.b])].to("cuda", torch.long)
            yield x, x


class View(Module):
    def __init__(s, k):
        super().__init__()
        s.k = k

    def forward(s, x):
        return x.view(-1, s.k)


class PyramidAutoEncoder(Sequential):
    def forward(s, x):
        n = int(np.log2(x.size(1)))
        t = []
        for _ in range(n):
            x = s[0](x)
            t.append(x)
        t = list(s[1](torch.cat(t)).split([len(t) for t in t]))
        x = None
        while t:
            x = s[2](t.pop() if x is None else torch.cat([x, t.pop()]))
        return x.view(n, -1, x.size(1))


class MultiCrossEntropyLoss(CrossEntropyLoss):
    def forward(s, x, t):
        return -x.log_softmax(2)[:, torch.arange(x.size(1), device="cuda"), t.flatten()].mean()


def Accuracy(i):
    def Accuracy(x, t):
        return accuracy(x[i], t)

    Accuracy.__name__ = str(int(i))
    return Accuracy

In [ ]:
m = Learner(
    DataLoaders(Dat(256, 9, 100), Dat(256, 20, 10000)),
    Sequential(
        Embedding(21482, 32),
        PyramidAutoEncoder(
            Sequential(View(64), Linear(64, 32, 0), BatchNorm1d(32, track_running_stats=0), Swish()),
            Sequential(Linear(32, 32, 0), BatchNorm1d(32, track_running_stats=0), Swish(), Linear(32, 32)),
            Sequential(BatchNorm1d(32, track_running_stats=0), Swish(), Linear(32, 64, 0), View(32)),
        ),
        Linear(32, 32, 0),
        BatchNorm1dFlat(32, track_running_stats=0),
        Swish(),
        Linear(32, 21482),
    ).cuda(),
    MultiCrossEntropyLoss(),
    metrics=map(Accuracy, range(8)),
    cbs=[ShowGraphCallback, SaveModelCallback(with_opt=1, reset_on_fit=0)],
)
m.lr_find()

In [ ]:
m.fit_one_cycle(500, _[0], 9, 9999, .01)

In [ ]:
m.save('einn5-autoencoder-Copy2')